# pySIPNET — Live Demo

A walkthrough of a complete SIPNET run in Python.

**Prerequisites:** SIPNET binary compiled (`make sipnet-standard` from the repo root).

For full details on any topic, see the [user guide](../docs/user-guide/).

---

**Sections**
1. [Climate data](#1.-Climate-data)
2. [Model parameters](#2.-Model-parameters)
3. [Running the model](#3.-Running-the-model)
4. [Inspecting outputs](#4.-Inspecting-outputs)
5. [Visualization](#5.-Visualization)
6. [File-backed climate for large workflows](#6.-File-backed-climate-for-large-workflows)

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pysipnet import ClimateDrivers, ClimateStaging, ModelPreset, SIPNETModel, SIPNETRunner
from pysipnet.parameters import (
    AllocationParams,
    InitialConditions,
    LeafPhysiologyParams,
    PhenologyParams,
    PhotosynthesisParams,
    RespirationParams,
    SIPNETParametersV1,
    WaterParams,
)

%matplotlib inline

---
## 1. Climate data

`ClimateDrivers` holds the meteorological forcing time series. Here we build two years of synthetic daily data from sinusoidal seasonal cycles — no external files needed for this demo.

In [ ]:
rng = np.random.default_rng(42)

n_years = 2
start_year = 2020

doys  = np.tile(np.arange(1, 366), n_years)
years = np.repeat(np.arange(start_year, start_year + n_years), 365)
theta = 2 * np.pi * (doys - 1) / 365

tair      = 7.5 + 12.5 * np.sin(theta - np.pi / 2) + rng.normal(0, 1.5, len(doys))
tsoil     = 7.5 + 10.0 * np.sin(theta - np.pi / 2 - 0.3) + rng.normal(0, 0.5, len(doys))
par       = np.maximum(0.0, 8 + 8 * np.sin(theta - np.pi / 2)) * np.maximum(0.5, 1 + rng.normal(0, 0.15, len(doys)))
precip    = rng.exponential(2.0, len(doys)) * (rng.random(len(doys)) < 0.3)
vpd       = np.maximum(50.0, 500 + 600 * np.maximum(0, np.sin(theta - np.pi / 2)) + rng.normal(0, 100, len(doys)))
vpd_soil  = np.maximum(50.0, vpd * 0.7 + rng.normal(0, 50, len(doys)))
vpress    = np.maximum(200.0, 1200 - vpd)
wspd      = np.maximum(0.1, rng.exponential(2.0, len(doys)) + 0.5)

climate = ClimateDrivers.from_dataframe(pd.DataFrame({
    "year": years, "day": doys, "time": 0.0, "length": 1.0,
    "tair": tair, "tsoil": tsoil, "par": par, "precip": precip,
    "vpd": vpd, "vpd_soil": vpd_soil, "vpress": vpress, "wspd": wspd,
}))

climate

In [ ]:
# The underlying DataFrame is always accessible
climate.data.head()

---
## 2. Model parameters

Parameters are organised into domain-specific groups and validated by Pydantic at construction time. Invalid values raise an error immediately — before anything touches the filesystem or the binary.

We use the `STANDARD` preset, which enables snow tracking, growing-degree-day phenology, and moisture-sensitive heterotrophic respiration.

In [ ]:
params = SIPNETParametersV1(
    initial_conditions=InitialConditions(
        plant_wood=30000.0,   # g C m⁻²
        lai=0.0,              # starts bare; GDD drives leaf-out
        soil=10000.0,         # g C m⁻²
        soil_water_frac=0.5,
        snow=1.0,             # cm water equiv.
        fine_root_frac=0.05,
        coarse_root_frac=0.15,
    ),
    photosynthesis=PhotosynthesisParams(
        a_max=112.0,          # nmol CO₂ g⁻¹ leaf s⁻¹
        a_max_frac=0.76,
        base_fol_resp_frac=0.1,
        psn_t_min=2.0,        # °C
        psn_t_opt=24.0,       # °C
        d_vpd_slope=0.05,
        d_vpd_exp=1.0,
        half_sat_par=300.0,   # mol m⁻² ground day⁻¹
        attenuation=0.5,
    ),
    phenology=PhenologyParams(
        gdd_leaf_on=100.0,    # °C·day accumulated from Jan 1
        leaf_off_day=270.0,   # DOY
        leaf_growth=50.0,     # g C m⁻²
        frac_leaf_fall=0.95,
        leaf_allocation=0.25,
        leaf_turnover_rate=1.0,
    ),
    respiration=RespirationParams(
        base_veg_resp=0.02,
        veg_resp_q10=2.0,
        growth_resp_frac=0.0,
        frozen_soil_fol_r_eff=0.5,
        frozen_soil_threshold=-1.0,
        base_fine_root_resp=0.5,
        base_coarse_root_resp=0.1,
        fine_root_q10=2.0,
        coarse_root_q10=2.0,
        base_soil_resp=0.06,
        soil_resp_q10=2.0,
        soil_resp_moist_effect=1.5,
    ),
    allocation=AllocationParams(
        fine_root_allocation=0.35,
        wood_allocation=0.30,
        fine_root_turnover_rate=1.0,
        coarse_root_turnover_rate=0.1,
        wood_turnover_rate=0.02,
    ),
    water=WaterParams(
        water_remove_frac=0.1,
        frozen_soil_eff=0.1,
        wue_const=10.0,
        soil_whc=12.0,
        litter_whc=5.0,
        immed_evap_frac=0.1,
        fast_flow_frac=0.1,
        snow_melt=0.15,       # cm °C⁻¹ day⁻¹
        rd_const=100.0,
        r_soil_const1=3.0,
        r_soil_const2=2.0,
    ),
    leaf=LeafPhysiologyParams(
        leaf_c_sp_wt=32.0,    # g C m⁻² leaf
        c_frac_leaf=0.45,
    ),
)

---
## 3. Running the model

### Low-level: `SIPNETRunner`

`SIPNETRunner.run()` writes all inputs to a fresh temporary directory, executes the SIPNET binary, parses the output, and cleans up. Each call is fully isolated — the same runner can be used for many runs in parallel.

In [ ]:
runner = SIPNETRunner(preset=ModelPreset.STANDARD)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")  # suppress low-VPD warnings from synthetic data
    result = runner.run(params, climate)

print(f"Success:    {result.provenance.success}")
print(f"Timesteps:  {len(result.outputs)}")
print(f"Run ID:     {result.provenance.run_id}")

### High-level: `SIPNETModel`

`SIPNETModel` wraps a runner and a baseline parameter set. You call it like a function, passing keyword overrides for any parameter you want to change — everything else stays at the baseline, and the baseline is never mutated.

This is the recommended interface for sensitivity analysis and ensemble workflows.

In [ ]:
model = SIPNETModel(runner, base_params=params, base_climate=climate)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result_baseline = model()                     # no overrides
    result_high_psn = model(a_max=150.0)          # higher photosynthetic capacity
    result_warm     = model(psn_t_opt=30.0)       # warmer optimum temperature
    result_combined = model(a_max=150.0, psn_t_opt=30.0)

In [ ]:
# Annual GPP across the four scenarios
scenarios = {
    "baseline":   result_baseline,
    "high a_max": result_high_psn,
    "warm T_opt": result_warm,
    "combined":   result_combined,
}

pd.DataFrame({
    name: res.outputs.groupby("year")["gpp"].sum()
    for name, res in scenarios.items()
})

---
## 4. Inspecting outputs

`result.outputs` is a pandas DataFrame with one row per timestep and one column per SIPNET output variable.

In [ ]:
result_baseline.outputs.head()

In [ ]:
print("All output columns:")
print(result_baseline.outputs.columns.tolist())

In [ ]:
# Annual summaries are just standard pandas groupby
result_baseline.outputs.groupby("year")[["nee", "gpp", "evapotranspiration"]].sum()

---
## 5. Visualization

In [ ]:
ts = result_baseline.outputs
x  = ts["year"] + (ts["day"] - 1) / 365

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(x, ts["nee"], label="NEE", color="steelblue", lw=0.9)
axes[0].plot(x, ts["gpp"], label="GPP", color="forestgreen", lw=0.9)
axes[0].axhline(0, color="k", lw=0.5, ls="--")
axes[0].set_ylabel("g C m\u207b\u00b2 per day")
axes[0].set_title("Carbon Fluxes")
axes[0].legend(frameon=False)

axes[1].plot(x, ts["evapotranspiration"], color="cornflowerblue", lw=0.9)
axes[1].set_ylabel("cm per day")
axes[1].set_title("Evapotranspiration")

axes[2].plot(x, ts["soil_c"],       label="Soil C",       color="sienna",    lw=0.9)
axes[2].plot(x, ts["plant_wood_c"], label="Plant Wood C", color="darkgreen", lw=0.9)
axes[2].set_ylabel("g C m\u207b\u00b2")
axes[2].set_xlabel("Year")
axes[2].set_title("Carbon Pools")
axes[2].legend(frameon=False)

fig.tight_layout()
plt.show()

In [ ]:
# GPP time series for all four scenarios
fig, ax = plt.subplots(figsize=(10, 4))

colors = ["steelblue", "forestgreen", "tomato", "darkorchid"]
for (name, res), color in zip(scenarios.items(), colors):
    ts_s = res.outputs
    x_s  = ts_s["year"] + (ts_s["day"] - 1) / 365
    ax.plot(x_s, ts_s["gpp"], label=name, color=color, lw=0.9, alpha=0.85)

ax.set_ylabel("GPP (g C m\u207b\u00b2 per day)")
ax.set_xlabel("Year")
ax.set_title("GPP \u2014 parameter sensitivity")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

---
## 6. File-backed climate for large workflows

In ensemble runs, each site typically has a pre-existing `.clim` file on disk. Loading every file into a DataFrame just to write it back out again for the SIPNET binary is wasteful. `ClimateDrivers.from_path()` creates a **file-backed** instance: the data is never read into memory unless you explicitly access `.data`. The runner stages it directly into the working directory — by default a copy, optionally a zero-I/O symlink.

In [ ]:
import tempfile
from pathlib import Path

# Simulate a pre-existing site file on disk
clim_dir  = Path(tempfile.mkdtemp())
clim_path = clim_dir / "site1.clim"

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    climate.to_file(clim_path)

print(f"Written to: {clim_path}")

In [ ]:
# File-backed: no data read yet
clim_ref = ClimateDrivers.from_path(clim_path, version="v1")

print(clim_ref)
print(f"Data in memory: {clim_ref._data is not None}")

In [ ]:
# Metadata is available from the lightweight peek — no full read needed
print(f"Timesteps:  {clim_ref.n_timesteps}")
print(f"Date range: {clim_ref.date_range}")
print(f"Still not loaded: {clim_ref._data is None}")

In [ ]:
# The runner copies the file directly — no read/write round-trip
# ClimateStaging.SYMLINK gives zero I/O overhead on Linux/macOS
runner_ref = SIPNETRunner(preset=ModelPreset.STANDARD, climate_staging=ClimateStaging.COPY)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result_ref = runner_ref.run(params, clim_ref)

print(f"Run succeeded:          {result_ref.provenance.success}")
print(f"Data still not loaded:  {clim_ref._data is None}")

Accessing `.data` on a file-backed instance loads and caches it — useful when you need to inspect or validate the file after the fact.

In [ ]:
clim_ref.data.head()

---

For more detail on any of these topics — parameter groups, ensemble workflows, file I/O options, output structure — see the [user guide](../docs/user-guide/).